# ASTAH Use Case Diagram -> XLSX

In [ ]:
from pathlib import Path

source = Path('AstahXMIExport.xml')
assert source.is_file(), f"Cannot find XMI source file {source.resolve()}"

xlsx_destination = Path('UseCaseList.xlsx')
assert xlsx_destination.parent.is_dir(), f"Missing parent directory {xlsx_destination.parent.resolve()}"

In [ ]:
from lxml import etree
import re
from urllib.parse import unquote

In [ ]:
def decode_astah_text(content: str) -> str:
    return unquote(content.replace('+', ' '))

In [ ]:
astah_uml = etree.parse(source)

nsmap = { 'UML': 'org.omg.xmi.namespace.UML' }

# Astah XMI structure

Beware: References use the same element tag.
Selecting with parent element 'UML:Namespace.ownedElement' excludes these references.


```
<UML:Namespace.ownedElement>
    <UML:UseCase xmi.id="3k5l-f6ca8f3017cec061a56206470e9f2635" name="EDI+Regel+anlegen">
    ...
    </UML:UseCase>
    
    <UML:Actor xmi.id="qdt-f6ca8f3017cec061a56206470e9f2635" name="EDI-ORGA+Onboarder+%28B2B%29">
    ...
    </UML:Actor>
    
    <UML:Association xmi.id="3nat-f6ca8f3017cec061a56206470e9f2635" name="" version="0" unSolvedFlag="false">
        <UML:ModelElement.namespace>
        <UML:Namespace xmi.idref="1odpj-f6ca8f3017cec061a56206470e9f2635"/>
        </UML:ModelElement.namespace>
        <UML:ModelElement.visibility xmi.value="public"/>
        <UML:Association.connection>
            <UML:AssociationEnd xmi.idref="3nav-f6ca8f3017cec061a56206470e9f2635"/>
            <UML:AssociationEnd xmi.idref="3nb1-f6ca8f3017cec061a56206470e9f2635"/>
        </UML:Association.connection>
    </UML:Association>
    
    <UML:AssociationEnd xmi.id="3nav-f6ca8f3017cec061a56206470e9f2635"
        <UML:AssociationEnd.participant>
                <UML:Classifier xmi.idref="qdt-f6ca8f3017cec061a56206470e9f2635"/>
              </UML:AssociationEnd.participant>
    
    <UML:AssociationEnd xmi.id="3nb1-f6ca8f3017cec061a56206470e9f2635"
        <UML:AssociationEnd.association>
            <UML:Association xmi.idref="3nat-f6ca8f3017cec061a56206470e9f2635"/>
    </UML:AssociationEnd.association>
</UML:Namespace.ownedElement>
```


## Collect entities

In [ ]:
usecases = list(astah_uml.findall('//{*}Namespace.ownedElement/{*}UseCase'))
usecases_map = { uc.attrib.get('xmi.id'): uc for uc in usecases }
len(usecases), len(usecases_map)

In [ ]:
for uc in usecases[:10]:
    print(f"{decode_astah_text(uc.get('name'))}")

In [ ]:
packages = list(astah_uml.findall('//{*}Namespace.ownedElement/{*}Package'))
packages_map = { pkg.attrib.get('xmi.id'): pkg for pkg in packages }
len(packages), len(packages_map)

In [ ]:
for pkg in list(packages_map.values())[:10]:
    print(f"{decode_astah_text(pkg.get('name'))} {pkg.attrib.get('xmi.id')}")

In [ ]:
actors = list(astah_uml.findall('//{*}Namespace.ownedElement/{*}Actor'))
#actor_map_v1 = dict(map(lambda a: (a.attrib.get('xmi.id'), a), actors))
actor_map = { a.attrib.get('xmi.id'): a for a in actors }
len(actors), len(actor_map)

In [ ]:
for actor in list(actor_map.values()):
    print(f"{decode_astah_text(actor.get('name'))}")

In [ ]:
def description(element: etree.Element) -> str:
    # Extract description text from XMI element ModelElement.definition
    # <UML:ModelElement.definition xmi.value=""
    #
    # :return: Description text found on the model element or None
    if element is None:
        return None
    definition = element.find("./UML:ModelElement.definition[ @xmi.value ]", nsmap)
    if definition is not None:
        return decode_astah_text(definition.attrib.get('xmi.value'))                            
    else:
        return None

## Collect relations/associations
Collect `UML:Association` and their `UML:AssociationEnd`

In [ ]:
associations = list(astah_uml.findall('//UML:Namespace.ownedElement/UML:Association[ @version ]', nsmap))
association_map = { a.attrib.get('xmi.id'): a for a in associations }
len(associations), len(association_map)

In [ ]:
association_ends = list(astah_uml.findall('//UML:Namespace.ownedElement/UML:AssociationEnd[ @version ]', nsmap))
association_end_map = { a.attrib.get('xmi.id'): a for a in association_ends }
len(association_ends), len(association_end_map)

In [ ]:
def resolve_association_end(end: etree.Element, association_end_lookup: dict) -> etree.Element:
    result = end
    reference_id = end.attrib.get('xmi.idref')
    if reference_id is not None:
        assert isinstance(reference_id, str)
        result = association_end_lookup.get(reference_id)
        assert result is not None
    assert result.get('xmi.id') is not None
    return result

In [ ]:
def dereference(end: etree.Element, lookup: dict) -> etree.Element:
    end_entity = end.find('./UML:Feature.owner/UML:Classifier', namespaces=nsmap)
    assert end_entity is not None
    idref = end_entity.attrib.get('xmi.idref')
    assert idref is not None
    return lookup.get(idref)

In [ ]:
def resolve_usecase(local: etree.Element, remote: etree.Element, use_case_map: dict) -> etree.Element:
    uc = dereference(local, use_case_map)
    if uc is None:
        uc = dereference(remote, use_case_map)
    return uc

In [ ]:
def resolve_actor(local: etree.Element, remote: etree.Element, actor_map: dict) -> etree.Element:
    actor = dereference(local, actor_map)
    if actor is None:
        actor = dereference(remote, actor_map)
    return actor

In [ ]:
def resolve_package(use_case: etree.Element, package_map: dict) -> etree.Element:
    parent_package = use_case.getparent().getparent()
    if parent_package.tag.endswith("Package"):
        identifier = parent_package.attrib.get('xmi.id')
        package = package_map.get(identifier)
        assert package is not None, f"No package for use case {use_case.attrib.get('xmi.id')}"
        return package
    else:
        # Use case is not located in a package. Parent is 'UML:Model'
        return None

In [ ]:
usecase_list = []

column_headers = [ 'Use Case', 'Use Case Description', 'Actor', 'Actor Description', 'System', 'System Description' ]

for association in associations:
    ends = list(association.findall('.//UML:Association.connection/UML:AssociationEnd', namespaces=nsmap))
    assert len(ends) == 2, f"Expecting exactly two ends for each association {association.attrib.get('xmi.id')}. Found {len(ends)}"

    local = resolve_association_end(ends[0], association_end_map)
    remote = resolve_association_end(ends[1], association_end_map)


    usecase = resolve_usecase(local, remote, usecases_map)
    if usecase is None:
        print(f"Cannot find usecase on relation {association.attrib.get('xmi.id')}")

    package = resolve_package(usecase, packages_map)
    if package is not None:
        package_name = decode_astah_text(package.get('name'))
    else:
        package_name = None
        
    actor = resolve_actor(local, remote, actor_map)
    if actor is None:
        print(f"Cannot find actor on relation {association.attrib.get('xmi.id')}")
    
    decode_astah_text(usecase.get('name'))


    # <UML:ModelElement.definition xmi.value="
    
    usecase_list.append( [ 
        decode_astah_text(usecase.get('name')),
        description(usecase),
        decode_astah_text(actor.get('name')),
        description(actor),
        package_name,
        description(package),
        ]
    )        


# Create Excel

In [ ]:
import xlsxwriter

In [ ]:
workbook = xlsxwriter.Workbook(xlsx_destination)

worksheet = workbook.add_worksheet('UseCases')
header_row_format = workbook.add_format({'font_color': '#FFFFFF', 'bold': True, 'bg_color': '#303030'})

col = 0
for header in column_headers:
    worksheet.write(0, col, header, header_row_format)
    col += 1

row = 1

for uc in usecase_list:
    col = 0
    for index, field in enumerate(uc):
        worksheet.write(row, index, field)
    col += len(uc)
    row += 1

worksheet.set_column(0, 0, 60)
worksheet.set_column(1, len(column_headers) - 1, 30)

worksheet.autofilter(0, 0, len(usecases), len(column_headers) - 1)

workbook.close()
print(f"Wrote {len(usecases)} to {xlsx_destination}")